# Presenting complaints by keyword dictionary (sensitivity analysis)

Sensitivity analysis for the way the presenting complaint is grouped. The paper groups complaints by clustering sentence embeddings (`../MIMICED/2_chiefcomp.ipynb`, k = 75). This notebook groups them instead with the published keyword dictionary of Coggan et al. (arXiv:2503.22781; repository hcoggan/mimic-iv-ed, script `mimic-chief-complaint.r`), which was written for this same dataset, and repeats the presenting-complaint table of the paper (Table 3) with those categories.

The dictionary is applied as published, with one change: its Transfer category is omitted because it records how the patient arrived rather than why. Each category is a list of search strings; a visit is flagged for a category when its upper-cased raw complaint contains any of the strings, so a visit can carry more than one category and the percentages in a column can sum to more than 100. Visits matching none of the 53 categories are counted as Uncategorised.

Inputs: `../MIMICED/ed_join_cc.parquet`. Output: `R7_coggan_table_S29.docx` (placeholder number; the supplement is renumbered at the end).

# Data

In [1]:
if(!require("pacman")) install.packages("pacman")
pacman::p_load(arrow, gt, tidyverse)

ed_data <- read_parquet(
    "../MIMICED/ed_join_cc.parquet",
    col_select = c(subject_id, stay_id, chiefcomplaint, acuity, age, race, nyu, res, acuity_out)
  ) |>
  mutate(combined_group = paste0(nyu, res, acuity_out))

# same eligibility filter as 3_descriptive.ipynb
analysis <- ed_data |>
  filter(!is.na(acuity) & !is.na(age) & !is.na(race))

cat(sprintf("Analysis sample: %s visits from %s patients\n", format(nrow(analysis), big.mark = ","), format(n_distinct(analysis$subject_id), big.mark = ",")))


Loading required package: pacman



Analysis sample: 417,723 visits from 201,106 patients


# The dictionary

The 53 categories and their search strings, copied from `mimic-chief-complaint.r` (Coggan et al.) without change apart from the omitted Transfer category. The five exact-string replacements that precede matching in the published script (HA, AH, OD, SI, ST) are applied as published. Names in `category_labels` are the spelled-out versions used in the table.

In [2]:
complaint_dict <- list(
    "abd_pain" = c("ABDOMINAL PAIN", "ABDO PAIN", "ABD PAIN", "ab PAIN", "EPIGASTRIC PAIN", "ABDOMINAL DISTENTION", "PELVIC PAIN", "RIB PAIN", "RLQ PAIN", "RUQ PAIN"),
    "chest_pain" = c("CHEST PAIN", "CP", "C/P"),
    "fall" = c("FALL"),
    "dyspnea" = c("DYSPNEA", "SOB", "SHORTNESS OF BREATH", "SHORT OF BREATH", "WHEEZING", "DIFFICULTY BREATHING", "RESPIRATORY DISTRESS"),
    "headache" = c("HEADACHE", "H/A", "HEAD ACHE", "MIGRAINE"),
    "fever" = c("FEVER"),
    "intox" = c("ETOH", "SUBSTANCE", "DETOX", "WITHDRAWAL"),
    "overdose" = c("OVERDOSE"),
    "nvd" = c("N/V", "NAUSEA", "DIARRHEA", "VOMITING"),
    "back_pain" = c("BACK", "LBP"),
    "weak_and_dizzy" = c("DIZZINESS", "WEAKNESS", "LIGHTHEADED", "SYNCOPE", "PRESYNCOPE", "FATIGUE", "LETHARGY", "DIZZY", "VERTIGO"),
    "wound" = c("WOUND", "LACERATION"),
    "suicidal" = c("SUICIDAL", "SELF HARM", "ATTEMPT"),
    "cough" = c("COUGH"),
    "abnormal_labs" = c("ABNORMAL", "ABNL"),
    "ili" = c("ILI", "INFLUENZA", "FLU SYMPTOMS", "FLU LIKE", "FLU-LIKE"),
    "confusion" = c("ALTERED MENTAL STATUS", "CONFUSION", "AGITATION"),
    "mental_illness" = c("ANXIETY", "DEPRESSION", "PSYCH", "HALLUCINATIONS", "A/H", "DELUSION", "MANIA"),
    "mvc" = c("MVC", "PED STRUCK", "BICYCLE ACCIDENT", "MOTORCYCLE ACCIDENT"),
    "head_injury" = c("HEAD INJURY", "HEAD BLEED", "HEADBLEED", "HEAD LAC"),
    "sore_throat" = c("SORE THROAT"),
    "palpitations" = c("PALPITATIONS", "TACHYCARDIA", "ABNORMAL EKG", "BRADYCARDIA", "FIBRILLATION"),
    "rectal_pain" = c("RECTAL PAIN"),
    "rectal_bleeding" = c("BRBPR", "RECTAL BLEEDING", "BLEEDING FROM RECTUM", "BLOOD IN STOOL", "BLOODY STOOL", "MELENA", "HEMATURIA", "BLOOD IN URINE", "HEMATEMESIS", "HEMOPTYSIS", "COFFEE GROUND EMESIS"),
    "neck_pain" = c("NECK"),
    "seizures" = c("SEIZURE"),
    "rash" = c("RASH"),
    "dysuria" = c("DYSURIA", "URINARY RETENTION", "URINARY FREQUENCY"),
    "flank_pain" = c("FLANK"),
    "walking_problems" = c("UNSTEADY GAIT", "UNABLE TO AMBULATE", "DIFFICULTY AMBULATING", "INABILITY TO AMBULATE", "DIFF AMBULATING", "CAN'T WALK", "CANT WALK", "UNABLE TO WALK", "VAGINAL SPOTTING"),
    "hypertension" = c("HYPERTENSION", "HIGH BLOOD PRESSURE"),
    "vaginal_bleeding" = c("VAGINAL BLEEDING", "VAG BLEEDING", "VAG BLEED", "VAGINAL BLEED", "VAGINAL PAIN", "VAGINAL SPOTTING",  "VAG SPOTTING"),
    "blood_sugar" = c("HYPERGLYCEMIA", "HYPOGLYCEMIA", "BLOOD SUGAR"),
    "allergy" = c("ALLERGIC", "ALLERGY"),
    "pregnant" = c("PREG", "LABOR"),
    "knee_pain" = c("KNEE"),
    "assault" = c("ASSAULT"),
    "low_blood_pressure" = c("HYPOTENSION", "LOW BLOOD PRESSURE"),
    "foot_pain" = c("FOOT", "ANKLE", "TOE"),
    "vision_issues" = c("VISION", "SEEING FLOATERS", "BLIND", "VISUAL"),
    "arm_pain" = c("ARM", "ELBOW"),
    "shoulder_pain" = c("SHOULDER"),
    "hip_pain" = c("HIP"),
    "abscess" = c("ABCESS"),
    "asthma" = c("ASTHMA"),
    "nosebleed" = c("EPISTAXIS"),
    "hypoxia" = c("HYPOXIA"),
    "hematoma" = c("ICH", "SDH", "SAH"),
    "leg_pain_or_swelling" = c("LEG", "CALF"),
    "eye_pain" = c("EYE PAIN"),
    "ear_pain" = c("EAR PAIN"),
    "numbness" = c("NUMBNESS"),
    "hand_finger_wrist" = c("HAND", "FINGER", "WRIST")
)

category_labels <- c(
  abd_pain = "Abdominal pain", chest_pain = "Chest pain", fall = "Fall", dyspnea = "Shortness of breath", headache = "Headache",
  fever = "Fever", intox = "Alcohol or substance intoxication", overdose = "Overdose", nvd = "Nausea, vomiting, diarrhea",
  back_pain = "Back pain", weak_and_dizzy = "Weakness, dizziness, syncope, fatigue", wound = "Wound or laceration",
  suicidal = "Suicidal ideation", cough = "Cough", abnormal_labs = "Abnormal laboratory results", ili = "Influenza-like illness",
  confusion = "Confusion or altered mental status", mental_illness = "Anxiety, depression, psychiatric", mvc = "Motor vehicle collision",
  head_injury = "Head injury", sore_throat = "Sore throat", palpitations = "Palpitations or abnormal heart rhythm", rectal_pain = "Rectal pain",
  rectal_bleeding = "Rectal, urinary or vomited blood", neck_pain = "Neck pain", seizures = "Seizure", rash = "Rash",
  dysuria = "Painful or difficult urination", flank_pain = "Flank pain", walking_problems = "Difficulty walking",
  hypertension = "High blood pressure", vaginal_bleeding = "Vaginal bleeding or pain", blood_sugar = "Abnormal blood sugar",
  allergy = "Allergic reaction", pregnant = "Pregnancy", knee_pain = "Knee pain", assault = "Assault", low_blood_pressure = "Low blood pressure",
  foot_pain = "Foot, ankle or toe pain", vision_issues = "Vision problems", arm_pain = "Arm or elbow pain", shoulder_pain = "Shoulder pain",
  hip_pain = "Hip pain", abscess = "Abscess", asthma = "Asthma", nosebleed = "Nosebleed", hypoxia = "Low oxygen", hematoma = "Brain hemorrhage",
  leg_pain_or_swelling = "Leg pain or swelling", eye_pain = "Eye pain", ear_pain = "Ear pain", numbness = "Numbness",
  hand_finger_wrist = "Hand, finger or wrist injury"
)
stopifnot(setequal(names(complaint_dict), names(category_labels)), length(complaint_dict) == 53)


# Apply the dictionary

In [3]:
cc <- toupper(coalesce(analysis$chiefcomplaint, ""))
# the published script's exact-string replacements, applied before matching
cc[cc == "HA"] <- "H/A"; cc[cc == "AH"] <- "A/H"; cc[cc == "OD"] <- "OVERDOSE"; cc[cc == "SI"] <- "SUICIDAL"; cc[cc == "ST"] <- "TACHYCARDIA"

flags <- map(complaint_dict, \(strings) {
  hit <- rep(FALSE, length(cc))
  for (s in strings) hit <- hit | grepl(s, cc, fixed = TRUE)   # grepl() in the published script is case-sensitive on upper-cased text
  hit
}) |>
  as_tibble()
flags$uncategorised <- !apply(flags, 1, any)
colnames(flags) <- c(unname(category_labels), "Uncategorised")

coverage <- tibble(
  measure = c("Visits with at least one category", "Visits with two or more categories", "Uncategorised (no match)", "Blank or redacted complaint"),
  n = c(sum(!flags$Uncategorised), sum(rowSums(flags[, 1:53]) >= 2), sum(flags$Uncategorised), sum(str_trim(coalesce(analysis$chiefcomplaint, "")) == "" | grepl("^_+$", coalesce(analysis$chiefcomplaint, "")))),
  pct = round(100 * n / nrow(analysis), 1)
)
coverage |> print()


# A tibble: 4 × 3
  measure                                 n   pct
  <chr>                               <int> <dbl>
1 Visits with at least one category  363128  86.9
2 Visits with two or more categories 102089  24.4
3 Uncategorised (no match)            54595  13.1
4 Blank or redacted complaint          1251   0.3


## Check

In [4]:
checks <- tibble(
  check = c("Analysis sample (visits)", "Analysis sample (patients)", "Flagged visits", "Categories", "Flag rows", "Uncategorised + any category = all"),
  observed = c(nrow(analysis), n_distinct(analysis$subject_id), sum(analysis$combined_group != "000"), ncol(flags) - 1, nrow(flags),
               sum(flags$Uncategorised) + sum(!flags$Uncategorised)),
  expected = c(417723L, 201106L, 209429L, 53L, 417723L, 417723L)
) |>
  mutate(result = if_else(observed == expected, "PASS", "FAIL"))
checks |> print()
stopifnot(all(checks$result == "PASS"))


# A tibble: 6 × 4
  check                              observed expected result
  <chr>                                 <dbl>    <int> <chr> 
1 Analysis sample (visits)             417723   417723 PASS  
2 Analysis sample (patients)           201106   201106 PASS  
3 Flagged visits                       209429   209429 PASS  
4 Categories                               53       53 PASS  
5 Flag rows                            417723   417723 PASS  
6 Uncategorised + any category = all   417723   417723 PASS  


# Table S29: Table 3 with the dictionary categories

The three most common categories in each exclusive algorithm combination, with the visits not identified by any algorithm first, as in the paper's Table 3. Percentages are of the visits in each column; a visit can be in more than one category.

In [5]:
combo_labels <- c(
  "000" = "Not identified by any algorithm", "100" = "Diagnosis", "010" = "Resources", "001" = "Triage Acuity",
  "110" = "Diagnosis+ Resources", "011" = "Resources+ Triage Acuity", "101" = "Diagnosis+ Triage Acuity", "111" = "All"
)

top_3 <- function(rows) {
  colMeans(flags[rows, ]) |>
    sort(decreasing = TRUE) |>
    head(3) |>
    (\(p) sprintf("%s (%.1f%%)", names(p), 100 * p))()
}

table_s29 <- bind_cols(
  tibble(Rank = c("1st", "2nd", "3rd")),
  as_tibble(map(set_names(names(combo_labels), combo_labels), \(g) top_3(analysis$combined_group == g)))
)
table_s29 |> print(width = Inf)

# the share of each column with no category, for the text
uncat_by_group <- map_dbl(names(combo_labels), \(g) 100 * mean(flags$Uncategorised[analysis$combined_group == g])) |>
  set_names(combo_labels) |>
  round(1)
uncat_by_group |> print()

note <- paste(
  sprintf("N = %s visits from %s patients.", format(nrow(analysis), big.mark = ","), format(n_distinct(analysis$subject_id), big.mark = ",")),
  "Columns are exclusive algorithm combinations, with the visits not identified by any algorithm first; percentages are of the visits in each column.",
  "Complaints are categories from the keyword dictionary of Coggan et al. (arXiv:2503.22781), applied as published except that the Transfer category is omitted; a visit can carry more than one category, so percentages can sum to more than 100.",
  sprintf("Uncategorised is a visit matching none of the 53 categories (%.1f%% of the sample). The paper groups complaints by clustering (Table 3).", 100 * mean(flags$Uncategorised))
)

tbl_gt <- table_s29 |>
  gt() |>
  tab_header(title = "Supplemental Table S29. Most common presenting complaints by combination of algorithms, using the Coggan et al. keyword dictionary (sensitivity analysis)") |>
  tab_spanner(label = "Potentially Avoidable ED Visits (by algorithm, % of visits)", columns = -c(Rank, `Not identified by any algorithm`)) |>
  cols_align(align = "center", columns = -Rank) |>
  tab_source_note(source_note = note)

gtsave(tbl_gt, filename = "R7_coggan_table_S29.docx")
tbl_gt


# A tibble: 3 × 9
  Rank  `Not identified by any algorithm`           
  <chr> <chr>                                       
1 1st   Uncategorised (13.6%)                       
2 2nd   Abdominal pain (11.9%)                      
3 3rd   Weakness, dizziness, syncope, fatigue (8.6%)
  Diagnosis                         
  <chr>                             
1 Abdominal pain (21.1%)            
2 Nausea, vomiting, diarrhea (12.6%)
3 Uncategorised (9.3%)              
  Resources                                   
  <chr>                                       
1 Uncategorised (14.6%)                       
2 Alcohol or substance intoxication (13.8%)   
3 Weakness, dizziness, syncope, fatigue (9.5%)
  `Triage Acuity`                     
  <chr>                               
1 Uncategorised (17.7%)               
2 Hand, finger or wrist injury (16.0%)
3 Foot, ankle or toe pain (15.7%)     
  `Diagnosis+ Resources`                       
  <chr>                                        
1 Abdo

Not identified by any algorithm                       Diagnosis 
                           13.6                             9.3 
                      Resources                   Triage Acuity 
                           14.6                            17.7 
           Diagnosis+ Resources        Resources+ Triage Acuity 
                           11.3                            22.4 
       Diagnosis+ Triage Acuity                             All 
                           20.1                            31.0 


ERROR while rich displaying an object: Error in (function (..., row.names = NULL, check.rows = FALSE, check.names = TRUE, : arguments imply differing number of rows: 1, 0, 10

Traceback:
1. sapply(x, f, simplify = simplify)
2. lapply(X = X, FUN = FUN, ...)
3. FUN(X[[i]], ...)
4. tryCatch(withCallingHandlers({
 .     if (!mime %in% names(repr::mime2repr)) 
 .         stop("No repr_* for mimetype ", mime, " in repr::mime2repr")
 .     rpr <- repr::mime2repr[[mime]](obj)
 .     if (is.null(rpr)) 
 .         return(NULL)
 .     prepare_content(is.raw(rpr), rpr)
 . }, error = error_handler), error = outer_handler)
5. tryCatchList(expr, classes, parentenv, handlers)
6. tryCatchOne(expr, names, parentenv, handlers[[1L]])
7. doTryCatch(return(expr), name, parentenv, handler)
8. withCallingHandlers({
 .     if (!mime %in% names(repr::mime2repr)) 
 .         stop("No repr_* for mimetype ", mime, " in repr::mime2repr")
 .     rpr <- repr::mime2repr[[mime]](obj)
 .     if (is.null(rpr)) 
 .       

ERROR while rich displaying an object: Error in (function (..., row.names = NULL, check.rows = FALSE, check.names = TRUE, : arguments imply differing number of rows: 1, 0, 10

Traceback:
1. sapply(x, f, simplify = simplify)
2. lapply(X = X, FUN = FUN, ...)
3. FUN(X[[i]], ...)
4. tryCatch(withCallingHandlers({
 .     if (!mime %in% names(repr::mime2repr)) 
 .         stop("No repr_* for mimetype ", mime, " in repr::mime2repr")
 .     rpr <- repr::mime2repr[[mime]](obj)
 .     if (is.null(rpr)) 
 .         return(NULL)
 .     prepare_content(is.raw(rpr), rpr)
 . }, error = error_handler), error = outer_handler)
5. tryCatchList(expr, classes, parentenv, handlers)
6. tryCatchOne(expr, names, parentenv, handlers[[1L]])
7. doTryCatch(return(expr), name, parentenv, handler)
8. withCallingHandlers({
 .     if (!mime %in% names(repr::mime2repr)) 
 .         stop("No repr_* for mimetype ", mime, " in repr::mime2repr")
 .     rpr <- repr::mime2repr[[mime]](obj)
 .     if (is.null(rpr)) 
 .       

ERROR while rich displaying an object: Error in (function (..., row.names = NULL, check.rows = FALSE, check.names = TRUE, : arguments imply differing number of rows: 1, 0, 10

Traceback:
1. sapply(x, f, simplify = simplify)
2. lapply(X = X, FUN = FUN, ...)
3. FUN(X[[i]], ...)
4. tryCatch(withCallingHandlers({
 .     if (!mime %in% names(repr::mime2repr)) 
 .         stop("No repr_* for mimetype ", mime, " in repr::mime2repr")
 .     rpr <- repr::mime2repr[[mime]](obj)
 .     if (is.null(rpr)) 
 .         return(NULL)
 .     prepare_content(is.raw(rpr), rpr)
 . }, error = error_handler), error = outer_handler)
5. tryCatchList(expr, classes, parentenv, handlers)
6. tryCatchOne(expr, names, parentenv, handlers[[1L]])
7. doTryCatch(return(expr), name, parentenv, handler)
8. withCallingHandlers({
 .     if (!mime %in% names(repr::mime2repr)) 
 .         stop("No repr_* for mimetype ", mime, " in repr::mime2repr")
 .     rpr <- repr::mime2repr[[mime]](obj)
 .     if (is.null(rpr)) 
 .       

<div id="dpnhbihndf" style="padding-left:0px;padding-right:0px;padding-top:10px;padding-bottom:10px;overflow-x:auto;overflow-y:auto;width:auto;height:auto;">
  <style>#dpnhbihndf table {
  font-family: system-ui, 'Segoe UI', Roboto, Helvetica, Arial, sans-serif, 'Apple Color Emoji', 'Segoe UI Emoji', 'Segoe UI Symbol', 'Noto Color Emoji';
  -webkit-font-smoothing: antialiased;
  -moz-osx-font-smoothing: grayscale;
}

#dpnhbihndf thead, #dpnhbihndf tbody, #dpnhbihndf tfoot, #dpnhbihndf tr, #dpnhbihndf td, #dpnhbihndf th {
  border-style: none;
}

#dpnhbihndf p {
  margin: 0;
  padding: 0;
}

#dpnhbihndf .gt_table {
  display: table;
  border-collapse: collapse;
  line-height: normal;
  margin-left: auto;
  margin-right: auto;
  color: #333333;
  font-size: 16px;
  font-weight: normal;
  font-style: normal;
  background-color: #FFFFFF;
  width: auto;
  border-top-style: solid;
  border-top-width: 2px;
  border-top-color: #A8A8A8;
  border-right-style: none;
  border-right-width: 2px;
  b